# Explore here

In [7]:
# Your code here
import pandas as pd

# Cargamos los datos usando la ruta de la plantilla
# Usamos encoding='latin-1' porque los archivos de NBA suelen tener caracteres especiales
df = pd.read_csv("../data/raw/nba_players.csv", encoding='latin-1')

# Vemos las primeras filas para entender las columnas
print(f"Dataset cargado: {df.shape[0]} jugadores registrados.")
df.head()

Dataset cargado: 213 jugadores registrados.


,RANK,NAME,TEAM,POS,AGE,GP,MPG,USG%,TO%,FTA,...,APG,SPG,BPG,TPG,P+R,P+A,P+R+A,VI,ORtg,DRtg
0,1,Joel Embiid,Phi,C,30.2,6,41.4,35.7,15.8,78,...,5.7,1.2,1.5,4.2,43.8,38.7,49.5,12.2,117.1,108.0
1,2,Jalen Brunson,Nyk,G,27.8,13,39.8,36.4,9.3,120,...,7.5,0.8,0.2,2.7,35.7,39.8,43.2,9.3,114.8,114.7
2,3,Damian Lillard,Mil,G,33.9,4,39.1,31.4,10.0,38,...,5.0,1.0,0.0,2.3,34.5,36.3,39.5,8.2,127.6,115.7
3,4,Shai Gilgeous-Alexander,Okc,G,25.9,10,39.9,32.3,8.9,81,...,6.4,1.3,1.7,2.2,37.4,36.6,43.8,11.2,118.3,106.9
4,5,Tyrese Maxey,Phi,G,23.6,6,44.6,28.1,8.6,28,...,6.8,0.8,0.3,2.2,35.0,36.7,41.8,9.1,120.9,113.3


In [8]:
df.isnull().sum()

RANK     0
NAME     0
TEAM     0
POS      0
AGE      0
GP       0
MPG      0
USG%     0
TO%      0
FTA      0
FT%      0
2PA      0
2P%      0
3PA      0
3P%      0
eFG%     0
TS%      0
PPG      0
RPG      0
APG      0
SPG      0
BPG      0
TPG      0
P+R      0
P+A      0
P+R+A    0
VI       0
ORtg     0
DRtg     0
dtype: int64

In [9]:
# 1. Ver todas las columnas disponibles para elegir nuestras variables
print("Columnas disponibles:")
print(list(df.columns))
print("-" * 50)

# 2. Ver cuántas posiciones diferentes hay exactamente
print("Distribución de las posiciones a predecir:")
print(df['POS'].value_counts())
print("-" * 50)

# 3. Revisar si hay valores nulos (vacíos) en alguna columna
print("Valores nulos por columna:")
print(df.isnull().sum()[df.isnull().sum() > 0]) # Solo mostramos las que tienen nulos

Columnas disponibles:
['RANK', 'NAME', 'TEAM', 'POS', 'AGE', 'GP', 'MPG', 'USG%', 'TO%', 'FTA', 'FT%', '2PA', '2P%', '3PA', '3P%', 'eFG%', 'TS%', 'PPG', 'RPG', 'APG', 'SPG', 'BPG', 'TPG', 'P+R', 'P+A', 'P+R+A', 'VI', 'ORtg', 'DRtg']
--------------------------------------------------
Distribución de las posiciones a predecir:
POS
G      83
F      72
C      24
G-F    10
C-F     9
F-C     9
F-G     6
Name: count, dtype: int64
--------------------------------------------------
Valores nulos por columna:
Series([], dtype: int64)


In [10]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
import pickle

# 1. Simplificar las posiciones a solo 3 categorías principales: G, F, C
df['POS_LIMPIA'] = df['POS'].apply(lambda x: x[0])
print("Nuevas posiciones:", df['POS_LIMPIA'].unique())

# 2. Elegir las características (X) y lo que queremos predecir (y)
# Seleccionamos: Minutos, Triples Intentados, Porcentaje de Triples, Puntos, Rebotes, Asistencias, Robos y Tapones
features = ['MPG', '3PA', '3P%', 'PPG', 'RPG', 'APG', 'SPG', 'BPG']
X = df[features]
y = df['POS_LIMPIA']

# 3. Dividir los datos (80% para entrenar, 20% para probar)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Crear y Entrenar el modelo
modelo = DecisionTreeClassifier(max_depth=4, random_state=42)
modelo.fit(X_train, y_train)

# 5. Medir la precisión
predicciones = modelo.predict(X_test)
precision = accuracy_score(y_test, predicciones)
print(f"Precisión del modelo: {precision * 100:.2f}%")

# 6. Guardar el modelo en la carpeta requerida por la plantilla
ruta_modelo = '../models/nba_model.sav'
pickle.dump(modelo, open(ruta_modelo, 'wb'))
print(f"¡Modelo guardado exitosamente en {ruta_modelo}!")

Nuevas posiciones: <StringArray>
['C', 'G', 'F']
Length: 3, dtype: str
Precisión del modelo: 51.16%
¡Modelo guardado exitosamente en ../models/nba_model.sav!


Bad pipe message: %s [b'\x0bF\xd0\xbb,\xb7\x9d\xe2\xb3\xe3:\xe0f3@\xce"\x7f\x00\x00\x1a\x13\x05\x13\x04\x13\x01\x13\x02\x13\x03\xc0\xb4\xc0\xb5\x00\xc7\x00\xc6\xc0\xb2\xc0\xb0\xc0\xb3\xc0\xb1\x01\x00\x00E\x00\n\x00\x0e\x00\x0c\x00\x17\x00\x18\x00\x19\x00\x1d\x01\x00\x00)\x00\x0b\x00\x02\x01\x00\x00+\x00\x03\x02\x03\x04\x00', b'\x1c\x00\x1a\x04\x01\x06\x01\x04\x03\x06\x03\x08']
Bad pipe message: %s [b'\x06\x08\x07']
Bad pipe message: %s [b'\x08\t\x08\x0b\x02\x01\x02']
Bad pipe message: %s [b'O\x89?]\xc1\xab\x92\xb9:i\xdeh\x9c\xb4M\x12\xa5-\x00\x00\x80\x00\x1d\x00\x1c\xfe\xff\xfe\xfe\x00c\x00e\x00\x11\x00r\x00\x13\x00s\x002\x00@\x00\xa2\x00t\x008\x00j\x00\xa3\xc0B\xc0V\xc0C\xc0W\x00D\x00']
Bad pipe message: %s [b'Phx\xc0X\xf9\xafI\x80\xf8\xb1\x00\xb2\xab', b'\x13\xa3\x00\x00\x80\x00\x1d\x00\x1c\xfe\xff\xfe\xfe\x00c\x00e\x00\x11\x00r\x00\x13\x00s\x002\x00@\x00\xa2\x00t\x008\x00j\x00\xa3\xc0B\xc0V\xc0C\xc0W\x00D\x00\xbd\xc0\x80\x00\x87\x00\xc3\xc0\x81\x00\x12\x00f\x00\x99\x00\x8f\x00']
Bad